# Contrat d'entrée v2 — chiffrer chaque retrait avant de l'adopter

Le carnet 05 a montré que les champs facultatifs vides biaisent l'estimation centrale vers
le bas (« champ manquant = annonce bâclée = pas cher », appris des données — MNAR). Le remède
retenu supprime la cause : **tous les champs deviennent obligatoires**, et le contrat se
réduit aux variables qui portent le prix (plan
`docs/plans/2026-08-10-contrat-entree-v2.md`).

L'importance par permutation du carnet 04 désigne les candidats au retrait : `critair`
(0,0002), `ct_valide_jusqu_a` (−0,0005), `couleur` (0,0001) sont du bruit ; `puissance_fisc`
(0,045) est redondante avec la DIN (0,208) ; `portes` (0,011) et `places` (0,014) pèsent un
peu. S'y ajoute la fusion des états « hors service » et « à réparer » (prix médians proches,
1 300–2 000 €). Mais une importance n'est pas une ablation : ce carnet mesure le **coût réel
en MAE** de chaque écart, un par un puis cumulé — la règle du projet depuis le retrait de
`region` (`app/src/preparation.py` : chaque écart chiffré, jamais décidé au jugé).

**Protocole.** Validation croisée 5 plis sur le **jeu d'ajustement seul** (11 928 lignes,
mêmes splits `random_state=42` que `ml/src/entrainement.py` — la calibration et le test ne
servent jamais à choisir). Modèle : le central servi,
`HistGradientBoostingRegressor(loss="quantile", quantile=0.5)`. Métrique : MAE.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split

sys.path.insert(0, str(Path("../../src").resolve()))
from leboncoin import clean_leboncoin

PARQUET = Path("../../data/leboncoin-private/raw/annonces.parquet")
PREMIUM = Path("../../references/premium_brand.csv")

df, _ = clean_leboncoin(PARQUET, PREMIUM)
date_reference = pd.Timestamp(df["scraped_at"].max()).tz_localize(None)
df["age"] = (date_reference - pd.to_datetime(dict(year=df["annee"], month=1, day=1))).dt.days / 365.25

# Splits identiques a l'entrainement : seul le jeu d'AJUSTEMENT sert ici. La calibration et
# le test restent hors de portee — on ne choisit jamais un contrat sur les donnees qui
# serviront a le garantir puis a le juger.
df_train, _df_test = train_test_split(df, test_size=0.20, random_state=42)
df_fit, _df_cal = train_test_split(df_train, test_size=0.25, random_state=42)
df_fit = df_fit.copy()

# Frequence du modele exact, comptee sur le jeu d'ajustement — meme table pour toutes les
# variantes : les comparaisons se font a conditions strictement egales.
freq = df_fit["modele"].value_counts()
df_fit["modele_freq"] = df_fit["modele"].map(freq).fillna(0.0).astype("float64")

print(f"jeu d'ajustement : {len(df_fit)} lignes")


jeu d'ajustement : 11928 lignes


## 1. Les variantes

L'état arrive brut du nettoyage (8 crans déclarés leboncoin) ; chaque variante choisit son
regroupement. Le v1 est celui en vigueur (5 crans, `app/src/preparation.py`) ; le v2 fusionne
« hors service » et « à réparer » — le carnet 03 avait montré ces crans indiscernables en
prix (1 300 € / 1 500 €, contre 2 000 € pour « petites réparations », proches tous les trois).


In [2]:
ETATS_V1 = {  # 8 crans -> 5 (contrat en vigueur)
    "not_drivable": "1_hors_service", "damaged": "1_hors_service",
    "major_repairs_needed": "1_hors_service", "minor_repairs_needed": "2_a_reparer",
    "normal_wear_and_tear": "3_usure", "good_overall_condition": "4_bon",
    "undamaged": "4_bon", "excellent_condition": "5_excellent",
}
ETATS_V2 = {  # 8 crans -> 4 (fusion hors service + a reparer)
    "not_drivable": "1_a_reparer", "damaged": "1_a_reparer",
    "major_repairs_needed": "1_a_reparer", "minor_repairs_needed": "1_a_reparer",
    "normal_wear_and_tear": "2_usure", "good_overall_condition": "3_bon",
    "undamaged": "3_bon", "excellent_condition": "4_excellent",
}

NUM_V1 = ["age", "kilometrage", "puissance_din", "puissance_fisc", "portes", "places",
          "critair", "ct_valide_jusqu_a", "boite_auto", "niveau", "modele_freq"]
CAT_V1 = ["energie_grp", "marque", "couleur", "etat"]

NUM_V2 = ["age", "kilometrage", "puissance_din", "boite_auto", "niveau", "modele_freq"]
CAT_V2 = ["energie_grp", "marque", "etat"]

def sans(liste, *retraits):
    return [c for c in liste if c not in retraits]

VARIANTES = {
    "v1 (reference)":        (NUM_V1, CAT_V1, ETATS_V1),
    "- critair, ct":         (sans(NUM_V1, "critair", "ct_valide_jusqu_a"), CAT_V1, ETATS_V1),
    "- puissance_fisc":      (sans(NUM_V1, "puissance_fisc"), CAT_V1, ETATS_V1),
    "- couleur":             (NUM_V1, sans(CAT_V1, "couleur"), ETATS_V1),
    "- portes, places":      (sans(NUM_V1, "portes", "places"), CAT_V1, ETATS_V1),
    "etats 5 -> 4 crans":    (NUM_V1, CAT_V1, ETATS_V2),
    "v2 (tout cumule)":      (NUM_V2, CAT_V2, ETATS_V2),
}


## 2. La mesure

Même graine partout (plis et modèle) : la seule chose qui change d'une ligne à l'autre est
le jeu de variables. Les écarts sont donc imputables au contrat, pas au hasard.


In [3]:
def mae_cv(num, cat, etats):
    X = df_fit.reindex(columns=num).astype("float64")
    for c in cat:
        col = df_fit["etat"].map(etats) if c == "etat" else df_fit[c]
        X[c] = col.fillna("(inconnu)").astype("category")
    modele = HistGradientBoostingRegressor(
        loss="quantile", quantile=0.5, categorical_features=cat, random_state=42
    )
    plis = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(modele, X, df_fit["prix_eur"], cv=plis,
                             scoring="neg_mean_absolute_error", n_jobs=-1)
    return -scores.mean(), scores.std()

resultats = {}
for nom, (num, cat, etats) in VARIANTES.items():
    mae, ect = mae_cv(num, cat, etats)
    resultats[nom] = (mae, ect)
    print(f"{nom:<22} MAE {mae:7,.0f} EUR  (+/- {ect:,.0f})")

ref = resultats["v1 (reference)"][0]
print()
print("--- ecart par rapport au v1 (positif = le retrait coute) ---")
for nom, (mae, _) in resultats.items():
    if nom != "v1 (reference)":
        print(f"{nom:<22} {mae - ref:+7,.0f} EUR")


v1 (reference)         MAE   1,547 EUR  (+/- 8)


- critair, ct          MAE   1,543 EUR  (+/- 7)


- puissance_fisc       MAE   1,558 EUR  (+/- 16)


- couleur              MAE   1,533 EUR  (+/- 4)


- portes, places       MAE   1,561 EUR  (+/- 8)


etats 5 -> 4 crans     MAE   1,564 EUR  (+/- 17)


v2 (tout cumule)       MAE   1,590 EUR  (+/- 16)

--- ecart par rapport au v1 (positif = le retrait coute) ---
- critair, ct               -4 EUR
- puissance_fisc           +11 EUR
- couleur                  -15 EUR
- portes, places           +14 EUR
etats 5 -> 4 crans         +17 EUR
v2 (tout cumule)           +43 EUR


## 3. Lecture et décision

| écart | coût MAE (CV 5 plis) |
|---|---|
| − critair, ct | **−4 €** (retirer améliore) |
| − couleur | **−15 €** (retirer améliore) |
| − puissance_fisc | +11 € |
| − portes, places | +14 € |
| états 5 → 4 crans | +17 € |
| **v2 cumulé** | **+43 €** (± 16) |

Trois enseignements :

- `critair`, `ct_valide_jusqu_a` et `couleur` étaient bien du **bruit** : les retirer fait
  même légèrement mieux — l'importance par permutation ne s'était pas trompée ;
- les retraits « payants » (`puissance_fisc`, `portes`/`places`, fusion d'états) coûtent
  chacun 11 à 17 €, sous le seuil de 30 € fixé au plan ;
- le cumul, **+43 € de MAE**, est le prix du contrat simplifié. À mettre en regard de ce
  qu'il achète : la suppression du biais des champs vides mesuré au carnet 05 (**−506 €**
  de déplacement médian du central, jusqu'à −9 260 € sur le haut de gamme) et un formulaire
  de 8 champs qu'un vendeur peut réellement remplir en entier.

**Décision : contrat v2 adopté** — NUM = `age, kilometrage, puissance_din, boite_auto,
niveau, modele_freq` ; CAT = `energie_grp, marque, etat` (4 crans) ; tous les champs
obligatoires. Reporté dans `app/src/preparation.py` et l'ADR ML 0007 ; le modèle est
ré-entraîné puis re-calibré (nouveaux `Q`, couverture, largeur) et jugé sur le même test.